# Plotting Subgoal Proposer

In [ ]:
PATH = '/home/jennifer/scratch/aorl2/2026-03-16-00/2026-03-16-00.682e4de5697b9c301a2fa322d299ef547e570517f079b5019aacc5dfc5697c07/'
CKPT_NUM = 1000000

In [ ]:
import glob
import json
import os
import pathlib
import random
import signal
import sys
import time
from collections import defaultdict

import jax
import numpy as np
import tqdm
import wandb
import wandb.util
from absl import app, flags
from ml_collections import config_flags

from agents import agents
from utils.datasets import Dataset, GCDataset, HGCDataset, ReplayBuffer
from utils.flax_utils import restore_agent, save_agent
from utils.log_utils import CsvLogger, get_animal, get_exp_name, get_flag_dict, setup_wandb
from utils.plot_utils import plot_heatmap
from utils.samplers import to_oracle_rep
from utils.statistics import get_statistics_class
from wrappers.datafuncs_utils import clip_dataset, make_env_and_datasets
from utils.evaluation import evaluate_gcfql, evaluate
from ogbench.relabel_utils import add_oracle_reps, relabel_dataset

# Simplified setup - manually set the key flags
FLAGS = flags.FLAGS
FLAGS.env_name = 'humanoidmaze-medium-navigate-singletask-task3-v0'
FLAGS.dataset_dir = '/home/jennifer/scratch/data/humanoidmaze-medium-navigate-v0'
FLAGS.agent = {
    'agent_name': 'gcfql',
    'train_goal_proposer': True,
    'goal_proposer_type': 'low_actor_goals',
    'train_value': True,
    'horizon_length': 25,
    'action_chunking': False,
    'num_qs': 10,
    'q_agg': 'mean',
    'discount': 0.995,
    'batch_size': 256,
    'alpha': 30.0,
    'subgoal_steps': 25,
    'awr_invtemp': 0.0,
}
FLAGS.seed = 0

# Restore agent
agent_config = FLAGS.agent
agent = agents[agent_config['agent_name']].create(FLAGS.seed, None, agent_config)
agent = restore_agent(agent, PATH, CKPT_NUM)

# Create environment
datasets = [file for file in sorted(glob.glob(f'{FLAGS.dataset_dir}/*.npz')) if '-val.npz' not in file]
dataset_idx = 0
splits = FLAGS.env_name.split('-')
env_name = '-'.join(splits[:-3]) + '-v0'
env, train_dataset_data, val_dataset_data = make_env_and_datasets(
    env_name,
    dataset_path=datasets[dataset_idx],
    use_oracle_reps=True,
)

# Get a sample observation and goal
example_batch = Dataset.create(**train_dataset_data, freeze=False).sample(1)
observation = example_batch['observations'][0]
goal = example_batch['goals'][0]

# Sample 128 subgoals
rng = jax.random.PRNGKey(FLAGS.seed)
observations = np.repeat(observation[None], 128, axis=0)
goals = np.repeat(goal[None], 128, axis=0)
subgoals = np.asarray(agent.propose_goals(observations, goals, rng))

print(f"Sampled {len(subgoals)} subgoals")
print(f"Observation: {observation}")
print(f"Goal: {goal}")
print(f"First few subgoals: {subgoals[:5]}")

FileNotFoundError: [Errno 2] No such file or directory: '../../scratch/aorl2/2026-03-16-00/2026-03-16-00.0dfb77fdff1e9bb91c8f7575c8ad2c3f7d867f93572cfb2780acc46d86598233/flags.json'